══════════════════════════════════════════════════════════════
 WEEK 6  |  DAY 1  |  SQL BASICS
══════════════════════════════════════════════════════════════
 LEARNING OBJECTIVES
 ───────────────────
 1. Create a SQLite database and table entirely in Python using sqlite3
 2. Insert rows and query data with SELECT, WHERE, and ORDER BY
 3. Update and delete rows with WHERE conditions

 TIME:  ~30-35 minutes

 YOUTUBE
 ───────
 Search: "Python sqlite3 tutorial create table insert select"
 Search: "SQL UPDATE DELETE WHERE Python sqlite3"
══════════════════════════════════════════════════════════════


sqlite3 is part of the Python standard library — no pip install needed.
We use ":memory:" to run everything in RAM with no files created.
This is perfect for learning because it is self-contained and repeatable.


In [ ]:

import sqlite3



══════════════════════════════════════════════════════════════
 CONCEPT 1 — CREATING A SQLite DATABASE AND TABLE
══════════════════════════════════════════════════════════════
sqlite3 workflow:
  1. Connect:   conn = sqlite3.connect(":memory:")
  2. Get cursor: cur = conn.cursor()
  3. Execute SQL: cur.execute("SQL STATEMENT")
  4. Commit:    conn.commit()   (required after INSERT/UPDATE/DELETE)
  5. Close:     conn.close()

SQL DATA TYPES used in SQLite:
  INTEGER   -- whole numbers
  REAL      -- floating point
  TEXT      -- strings
  BLOB      -- raw binary data
  NULL      -- missing value

CREATE TABLE syntax:
  CREATE TABLE table_name (
      column_name datatype CONSTRAINT,
      ...
  );
Constraints: PRIMARY KEY, NOT NULL, UNIQUE, DEFAULT value


EXAMPLE ──────────────────────────────────────────────────────
Create an in-memory database


In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()



Create the employees table


In [ ]:
cur.execute("""
    CREATE TABLE employees (
        emp_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name        TEXT    NOT NULL,
        department  TEXT    NOT NULL,
        salary      REAL    NOT NULL,
        start_date  TEXT,
        is_active   INTEGER DEFAULT 1
    )
""")



Create the departments table


In [ ]:
cur.execute("""
    CREATE TABLE departments (
        dept_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        name      TEXT    NOT NULL UNIQUE,
        budget    REAL    DEFAULT 0.0,
        manager   TEXT
    )
""")

conn.commit()
print("Tables created successfully.")



Verify tables exist


In [ ]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cur.fetchall()
print("Tables in DB:", [t[0] for t in tables])   # ['employees', 'departments']




══════════════════════════════════════════════════════════════
 EXERCISE 1
══════════════════════════════════════════════════════════════
Using the same in-memory connection (conn, cur), create a table called "sales"
with these columns:
  sale_id     INTEGER PRIMARY KEY AUTOINCREMENT
  sale_date   TEXT NOT NULL
  product     TEXT NOT NULL
  quantity    INTEGER NOT NULL
  unit_price  REAL NOT NULL
  rep_name    TEXT

After creating it, verify it was created by querying sqlite_master.
Print: "Tables now in DB: ['employees', 'departments', 'sales']"

Expected output:
  Sales table created.
  Tables now in DB: ['employees', 'departments', 'sales']


══════════════════════════════════════════════════════════════
 CONCEPT 2 — INSERT, SELECT, WHERE, ORDER BY
══════════════════════════════════════════════════════════════
INSERT adds rows to a table.
SELECT retrieves rows from a table.

INSERT syntax:
  INSERT INTO table_name (col1, col2) VALUES (val1, val2);
Always use ? placeholders for values — NEVER build SQL strings with f-strings.
(f-string SQL is vulnerable to SQL injection attacks)

SELECT syntax:
  SELECT col1, col2 FROM table_name;
  SELECT * FROM table_name WHERE condition;
  SELECT * FROM table_name ORDER BY col DESC;
  SELECT * FROM table_name WHERE col = ? -- use ? for parameters

fetchall()  -- return all matching rows as a list of tuples
fetchone()  -- return the first matching row
fetchmany(n)-- return the next n rows


EXAMPLE ──────────────────────────────────────────────────────
Insert employee data using executemany (batch insert)


In [ ]:
employees_data = [
    ("Alice Ng",    "Engineering", 95000, "2021-03-15"),
    ("Bob Chen",    "Sales",       72000, "2019-07-01"),
    ("Carol Diaz",  "Finance",     81000, "2020-11-20"),
    ("Dave Park",   "Engineering", 88000, "2022-01-10"),
    ("Eve Torres",  "Sales",       67500, "2023-05-30"),
    ("Frank Wu",    "Finance",     79000, "2021-08-14"),
    ("Grace Lee",   "Engineering", 105000,"2018-04-22"),
]

cur.executemany(
    "INSERT INTO employees (name, department, salary, start_date) VALUES (?,?,?,?)",
    employees_data,
)
conn.commit()
print(f"\nInserted {cur.rowcount} employee(s).")



SELECT all employees


In [ ]:
print("\n=== All Employees ===")
cur.execute("SELECT emp_id, name, department, salary FROM employees ORDER BY emp_id")
for row in cur.fetchall():
    emp_id, name, dept, salary = row
    print(f"  [{emp_id}] {name:<15} | {dept:<15} | ${salary:>9,.2f}")



SELECT with WHERE


In [ ]:
print("\n=== Engineering Department ===")
cur.execute("""
    SELECT name, salary
    FROM   employees
    WHERE  department = ?
    ORDER BY salary DESC
""", ("Engineering",))

for name, salary in cur.fetchall():
    print(f"  {name}: ${salary:,.2f}")



Aggregate functions


In [ ]:
print("\n=== Salary Statistics ===")
cur.execute("""
    SELECT
        department,
        COUNT(*)            AS headcount,
        ROUND(AVG(salary),2) AS avg_salary,
        MIN(salary)         AS min_salary,
        MAX(salary)         AS max_salary
    FROM employees
    GROUP BY department
    ORDER BY avg_salary DESC
""")

rows = cur.fetchall()
print(f"{'Department':<15} {'Count':>6} {'Avg Salary':>12} {'Min':>10} {'Max':>10}")
print("-" * 58)
for dept, count, avg, low, high in rows:
    print(f"{dept:<15} {count:>6} ${avg:>11,.2f} ${low:>9,.2f} ${high:>9,.2f}")




══════════════════════════════════════════════════════════════
 EXERCISE 2
══════════════════════════════════════════════════════════════
Insert these rows into the "sales" table you created in Exercise 1:
  ("2024-01-15", "Laptop",   3, 899.99, "Bob Chen")
  ("2024-01-16", "Monitor",  5, 349.99, "Eve Torres")
  ("2024-01-17", "Keyboard", 10, 89.99, "Bob Chen")
  ("2024-01-18", "Laptop",   1, 899.99, "Carol Diaz")
  ("2024-01-19", "Mouse",   20, 39.99,  "Eve Torres")

Then:
A. SELECT all rows and print them
B. SELECT only rows where unit_price > 100, print them
C. SELECT and print total quantity and total revenue (quantity*unit_price)
   for each product, ordered by total revenue descending

Expected output (part C):
  Product   | Total Qty | Total Revenue
  Laptop    |     4     |   $3,599.96
  Monitor   |     5     |   $1,749.95
  Keyboard  |    10     |     $899.90
  Mouse     |    20     |     $799.80


══════════════════════════════════════════════════════════════
 CONCEPT 3 — UPDATE AND DELETE WITH WHERE
══════════════════════════════════════════════════════════════
UPDATE changes existing row values.
DELETE removes rows from the table.
Always use WHERE — without it, UPDATE and DELETE affect EVERY row.

UPDATE syntax:
  UPDATE table_name SET col1 = val1, col2 = val2 WHERE condition;

DELETE syntax:
  DELETE FROM table_name WHERE condition;

After UPDATE or DELETE, always commit:
  conn.commit()


EXAMPLE ──────────────────────────────────────────────────────


In [ ]:
print("\n=== BEFORE UPDATE ===")
cur.execute("SELECT name, salary, is_active FROM employees WHERE department = 'Sales'")
for row in cur.fetchall():
    print(f"  {row[0]}: ${row[1]:,.2f}  active={row[2]}")



Give all Sales employees a 10% raise


In [ ]:
cur.execute("""
    UPDATE employees
    SET    salary = ROUND(salary * 1.10, 2)
    WHERE  department = 'Sales'
""")
conn.commit()
print(f"Rows updated: {cur.rowcount}")

print("\n=== AFTER UPDATE (Sales +10%) ===")
cur.execute("SELECT name, salary FROM employees WHERE department = 'Sales'")
for row in cur.fetchall():
    print(f"  {row[0]}: ${row[1]:,.2f}")



Soft-delete: mark an employee as inactive (preferred over hard delete)


In [ ]:
cur.execute("UPDATE employees SET is_active = 0 WHERE name = ?", ("Frank Wu",))
conn.commit()
print(f"\nSoft-deleted Frank Wu. Rows affected: {cur.rowcount}")



Verify


In [ ]:
cur.execute("SELECT name, is_active FROM employees")
for name, active in cur.fetchall():
    status = "active" if active else "INACTIVE"
    print(f"  {name}: {status}")



Hard delete a record


In [ ]:
cur.execute("DELETE FROM employees WHERE name = ? AND is_active = 0", ("Frank Wu",))
conn.commit()
print(f"\nHard-deleted inactive employees. Rows removed: {cur.rowcount}")

cur.execute("SELECT COUNT(*) FROM employees")
print(f"Remaining employees: {cur.fetchone()[0]}")




══════════════════════════════════════════════════════════════
 EXERCISE 3
══════════════════════════════════════════════════════════════
Using the "sales" table from Exercise 2:

A. Update the unit_price of "Laptop" to 849.99 (price reduction)
   Print: "Updated X laptop price records"

B. Delete all sales where quantity > 15
   Print: "Deleted X bulk order records"

C. After the changes, SELECT all remaining rows and print them

D. Add a new column "is_commission_paid" using ALTER TABLE:
   ALTER TABLE sales ADD COLUMN is_commission_paid INTEGER DEFAULT 0
   Then update all rows where rep_name = "Bob Chen" to set is_commission_paid = 1
   Print how many of Bob's sales were updated

Expected output:
  Updated 2 laptop price records
  Deleted 1 bulk order records
  [remaining rows printed]
  Updated 2 Bob Chen commission records


Always close the connection when done


In [ ]:
conn.close()
print("\nDatabase connection closed.")


══════════════════════════════════════════════════════════════
 CONCEPT 4 — SQL SAFETY: PARAMETERIZED QUERIES
══════════════════════════════════════════════════════════════
 ROYAL ROAD STANDARD — W6
 Never build SQL strings with f-strings or string concatenation.
 Always use ? placeholders and pass values separately.
 This prevents SQL injection — one of the most common and destructive attacks.

 SQL injection: when a user-supplied value contains SQL syntax that the
 database executes as a command. Example attack value:
   "'; DROP TABLE employees; --"
 If you used an f-string, this would delete your table.

 Parameterized queries prevent this because the database driver treats
 the value as a literal string — never as SQL code.

 WRONG (vulnerable):
   query = f"SELECT * FROM users WHERE name = '{name}'"   # never do this

 RIGHT (safe):
   cur.execute("SELECT * FROM users WHERE name = ?", (name,))

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)")
cur.executemany("INSERT INTO products VALUES (?,?,?)", [
    (1, "Laptop",   4500.0),
    (2, "Monitor",  350.0),
    (3, "Keyboard", 89.0),
])
conn.commit()


# SAFE: parameterized query — value is never interpreted as SQL
def search_by_name(name):
    cur.execute("SELECT id, name, price FROM products WHERE name = ?", (name,))
    return cur.fetchall()


def update_price(product_id, new_price):
    cur.execute("UPDATE products SET price = ? WHERE id = ?", (new_price, product_id))
    conn.commit()
    return cur.rowcount


# Normal use
print(search_by_name("Laptop"))      # [(1, 'Laptop', 4500.0)]

# SQL injection attempt — treated as a literal string, returns nothing
attack = "' OR '1'='1"
result = search_by_name(attack)
print(f"Injection attempt returned: {result}")    # [] — database is safe

# Verify the table is untouched
cur.execute("SELECT COUNT(*) FROM products")
print(f"Products in table: {cur.fetchone()[0]}")  # 3

conn.close()

══════════════════════════════════════════════════════════════
 EXERCISE 4
══════════════════════════════════════════════════════════════
 Using the starting data below, create an in-memory SQLite database
 with a "users" table: (user_id INTEGER PK, username TEXT, role TEXT, salary REAL).
 Insert all 4 rows from users_data.

 Write two functions using parameterized queries only:
   get_users_by_role(conn, role) -> list of tuples
   update_salary(conn, username, new_salary) -> number of rows updated

 Call them as follows and print results:
   1. Get all users with role "engineer" — print the list
   2. Update salary for "bob" to 78000 — print "Updated: N row"
   3. Try get_users_by_role with attack_value below — print the result

 Expected output:
     Engineers: [(1, 'alice', 'engineer', 95000.0), (3, 'carol', 'engineer', 88000.0)]
     Updated: 1 row
     Injection attempt returned: [] rows

 --- starting data ---

In [ ]:
conn2 = sqlite3.connect(":memory:")
users_data = [
    (1, "alice", "engineer", 95000.0),
    (2, "bob",   "analyst",  72000.0),
    (3, "carol", "engineer", 88000.0),
    (4, "dave",  "manager",  105000.0),
]
attack_value = "' OR '1'='1"

══════════════════════════════════════════════════════════════
 CONCEPT 6 — TESTING YOUR PARAMETERIZED QUERY CONTRACT
══════════════════════════════════════════════════════════════

 Shift-Left Testing applied to SQL: verify that your query functions
 are actually parameterized — not just functionally correct on valid input.

 Two assertions a safe SQL function must satisfy:
   1. Returns correct data on valid input
   2. Returns empty results (not all rows) on a SQL injection attempt

 The second check is what separates a test from a smoke test.
 An f-string query also passes check 1 — only check 2 catches the vulnerability.

 Also assert that the table is intact after an injection attempt:
   cur.execute("SELECT COUNT(*) FROM table")
   assert cur.fetchone()[0] == expected_count

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import sqlite3

# Fresh in-memory database for this test block
conn_ex = sqlite3.connect(":memory:")
cur_ex  = conn_ex.cursor()
cur_ex.execute("CREATE TABLE items (id INTEGER PRIMARY KEY, name TEXT, category TEXT)")
cur_ex.executemany("INSERT INTO items VALUES (?,?,?)", [
    (1, "Laptop",   "Electronics"),
    (2, "Monitor",  "Electronics"),
    (3, "Desk",     "Furniture"),
])
conn_ex.commit()


def find_by_category(conn, category):
    c = conn.cursor()
    c.execute("SELECT id, name FROM items WHERE category = ?", (category,))
    return c.fetchall()


# 1. Assert valid input returns correct rows
result = find_by_category(conn_ex, "Electronics")
assert len(result) == 2,             "Electronics must return exactly 2 rows"
assert result[0][1] == "Laptop",     "first Electronics item must be Laptop"

# 2. Assert injection attempt returns zero rows (not all 3)
attack = "' OR '1'='1"
injected = find_by_category(conn_ex, attack)
assert len(injected) == 0,           "injection attack must return 0 rows, not all rows"

# 3. Assert table is untouched after injection attempt
cur_ex.execute("SELECT COUNT(*) FROM items")
assert cur_ex.fetchone()[0] == 3,    "injection must not alter or destroy the table"

conn_ex.close()
print("Parameterized query contract: all assertions passed.")

══════════════════════════════════════════════════════════════
 EXERCISE 6
══════════════════════════════════════════════════════════════

 conn_test, the "contracts" table, and get_by_status() are defined
 in the starting data below.

 Write assert statements to test the contract:
   1. get_by_status(conn_test, "active") returns exactly 2 rows
   2. The first row's client name is "Acme Corp"
   3. A SQL injection attempt returns 0 rows
   4. After the injection attempt, the table still has 3 rows total

 The last line must print: "Contracts query contract: all assertions passed."

 Expected output:
     Contracts query contract: all assertions passed.

 --- starting data ---

In [ ]:
import sqlite3

conn_test = sqlite3.connect(":memory:")
cur_test  = conn_test.cursor()
cur_test.execute("""
    CREATE TABLE contracts (
        id     INTEGER PRIMARY KEY,
        client TEXT,
        value  REAL,
        status TEXT
    )
""")
cur_test.executemany("INSERT INTO contracts VALUES (?,?,?,?)", [
    (1, "Acme Corp",   120000.0, "active"),
    (2, "Beta Ltd",     85000.0, "expired"),
    (3, "Gamma Inc",    97500.0, "active"),
])
conn_test.commit()


def get_by_status(conn, status):
    c = conn.cursor()
    c.execute("SELECT id, client, value FROM contracts WHERE status = ?", (status,))
    return c.fetchall()


sql_attack = "' OR '1'='1"




